# Violence Detection - Model Evaluation
Évaluation détaillée du modèle entraîné

In [ ]:
import numpy as np
import pickle
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from config import *

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# Load data
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']

# Split for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Test set: {len(X_test)} samples')
print(f'Violence: {np.sum(y_test == 1)}')
print(f'Non-Violence: {np.sum(y_test == 0)}')

In [ ]:
# Load model
model = tf.keras.models.load_model(f'{MODELS_DIR}/final_model.h5')
print('✓ Model loaded')

In [ ]:
# Predictions
print('Making predictions...')
y_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

print('✓ Predictions complete')

In [ ]:
# Classification report
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=CLASSES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{LOGS_DIR}/confusion_matrix.png')
plt.show()

# Normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Normalized Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{LOGS_DIR}/confusion_matrix_normalized.png')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig(f'{LOGS_DIR}/roc_curve.png')
plt.show()

print(f'AUC Score: {roc_auc:.4f}')

In [ ]:
# Prediction distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Violence class
violence_preds = y_pred_proba[y_test == 1]
axes[0].hist(violence_preds, bins=50, alpha=0.7, color='red', edgecolor='black')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=2)
axes[0].set_title('Prediction Distribution - Violence Class')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Frequency')

# Non-Violence class
non_violence_preds = y_pred_proba[y_test == 0]
axes[1].hist(non_violence_preds, bins=50, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=2)
axes[1].set_title('Prediction Distribution - Non-Violence Class')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(f'{LOGS_DIR}/prediction_distribution.png')
plt.show()

In [ ]:
# Analyze misclassifications
misclassified_idx = np.where(y_pred != y_test)[0]

print(f'\nMisclassified samples: {len(misclassified_idx)} ({len(misclassified_idx)/len(y_test)*100:.2f}%)')

if len(misclassified_idx) > 0:
    print('\nMisclassification details:')
    for idx in misclassified_idx[:10]:  # Show first 10
        true_label = CLASSES[y_test[idx]]
        pred_label = CLASSES[y_pred[idx]]
        confidence = y_pred_proba[idx][0]
        print(f'Sample {idx}: True={true_label}, Pred={pred_label}, Confidence={confidence:.3f}')